# Прогноз согласия клиента на срочный вклад

**Источник данных:** телефонные маркетинговые кампании португальского банка ([Moro et al., 2011](http://hdl.handle.net/1822/14838)).

**Объём:** 45 211 записей о звонках клиентам (май 2008 — ноябрь 2010). Пропусков в данных нет.

---

### Задача

По данным о клиенте и параметрах кампании предсказать, **согласится ли клиент открыть срочный вклад**.

- **Ответ «да»** — клиент согласился (`yes`), около 12% случаев.
- **Ответ «нет»** — клиент отказался (`no`), около 88% случаев.

Признак **длительности звонка** (`duration`) в модель не включаем: эта информация появляется только во время разговора. Если использовать её заранее, модель будет «подглядывать» в результат — это называется **утечка данных**.

---

### Как оцениваем качество

Из-за дисбаланса классов доля правильных ответов (accuracy) не подходит: модель, которая всегда говорит «нет», уже даст ~88%.

Поэтому смотрим:
- **PR-AUC** — насколько хорошо модель находит редких клиентов, которые согласились;
- **Recall** — какую долю согласившихся мы не пропустили;
- **Precision** — насколько можно доверять предсказанию «да»;
- **F1** — баланс между recall и precision.

---

### Ход работы

1. Загрузка и первичный анализ данных
2. Подготовка признаков и проверка, что из этого реально помогает
3. Разделение на обучающую, проверочную и тестовую выборки
4. Обучение и сравнение четырёх вариантов логистической регрессии
5. Оценка качества и интерпретация результатов

---

### Краткие итоги

- Согласий мало (~12%), поэтому главная метрика — **PR-AUC**, а не accuracy.
- Создан признак **«был ли контакт раньше»** — вместо неоднозначного значения `pdays = -1`.
- Лучший вариант подготовки данных: **StandardScaler + новый признак** (по проверочной выборке).
- Данные разделены **по времени** (60% / 20% / 20%), потому что записи упорядочены по дате.
- Лучшая модель на проверочной выборке — **логистическая регрессия с L2-регуляризацией**.
- На тесте: PR-AUC ≈ **0.39**, recall ≈ **55%**, precision ≈ **38%**.
- Сильнее всего на решение влияют **месяц звонка**, **семейное положение** и **канал связи**.

---

### Описание признаков

**О клиенте:** возраст, род занятий, семейное положение, образование, просрочка по кредиту, баланс на счёте, ипотека, персональный кредит.

**О текущем звонке:** канал связи, день и месяц контакта, длительность разговора (только для анализа, не для модели).

**О кампании:** число звонков клиенту, дней с прошлого контакта, число прошлых контактов, исход прошлой кампании.

**Целевая переменная:** согласие на открытие срочного вклада (`yes` / `no`).


## 1. Загрузка данных

Читаем ARFF через `scipy.io.arff`, переименовываем колонки `V1`…`V16` в осмысленные имена и приводим типы к удобному для pandas виду.

In [94]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import arff
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42


In [ ]:
def resolve_data_path(filename: str = 'bank.arff') -> Path:
    """Ищем bank.arff рядом с ноутбуком и в типичных путях репозитория."""
    candidates = []

    # Cursor / VS Code знают путь к открытому .ipynb
    for key in ('__vsc_ipynb_file__', '__file__'):
        ref = globals().get(key)
        if ref:
            candidates.append(Path(ref).resolve().parent / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / filename,
        cwd / 'project' / filename,
    ])

    # Поднимаемся по дереву каталогов (если cwd не там, где ожидаем)
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / filename)
        candidates.append(parent / 'project' / filename)

    checked = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        checked.append(resolved)
        if resolved.exists():
            return resolved

    msg = (
        f'Файл {filename} не найден.\n'
        f'Текущая папка ядра: {cwd}\n'
        'Убедитесь, что в репозитории есть project/bank.arff '
        '(git pull / git lfs pull, если используете LFS).\n'
        'Проверенные пути:\n' + '\n'.join(f'  - {p}' for p in checked[:12])
    )
    raise FileNotFoundError(msg)


DATA_PATH = resolve_data_path()
print(f'Файл данных: {DATA_PATH}')

COLUMN_NAMES = {
    'V1': 'age',
    'V2': 'job',
    'V3': 'marital',
    'V4': 'education',
    'V5': 'default',
    'V6': 'balance',
    'V7': 'housing',
    'V8': 'loan',
    'V9': 'contact',
    'V10': 'day',
    'V11': 'month',
    'V12': 'duration',
    'V13': 'campaign',
    'V14': 'pdays',
    'V15': 'previous',
    'V16': 'poutcome',
    'Class': 'y',
}

CATEGORICAL_COLUMNS = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome', 'y',
]

raw_data, _ = arff.loadarff(DATA_PATH)
df = pd.DataFrame(raw_data)

# В ARFF строковые значения приходят как bytes — декодируем.
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.decode('utf-8')

df = df.rename(columns=COLUMN_NAMES)

# Class в ARFF: 1 = no, 2 = yes
df['y'] = df['y'].map({'1': 'no', '2': 'yes'})

for col in CATEGORICAL_COLUMNS:
    df[col] = df[col].astype('category')

INTEGER_COLUMNS = ['age', 'day', 'duration', 'campaign', 'pdays', 'previous']
df[INTEGER_COLUMNS] = df[INTEGER_COLUMNS].astype('int64')
df['balance'] = df['balance'].astype('int64')


In [ ]:
print(f'Загружено {df.shape[0]} строк и {df.shape[1]} колонок')
print(f'Доля согласий на срочный вклад: {(df["y"] == "yes").mean():.1%}')

df.head()


Загружено 45211 строк и 17 колонок
Доля согласий на срочный вклад: 11.7%


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [ ]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   age        45211 non-null  int64   
 1   job        45211 non-null  category
 2   marital    45211 non-null  category
 3   education  45211 non-null  category
 4   default    45211 non-null  category
 5   balance    45211 non-null  int64   
 6   housing    45211 non-null  category
 7   loan       45211 non-null  category
 8   contact    45211 non-null  category
 9   day        45211 non-null  int64   
 10  month      45211 non-null  category
 11  duration   45211 non-null  int64   
 12  campaign   45211 non-null  int64   
 13  pdays      45211 non-null  int64   
 14  previous   45211 non-null  int64   
 15  poutcome   45211 non-null  category
 16  y          45211 non-null  category
dtypes: category(10), int64(7)
memory usage: 2.8 MB


## 2. Первичный анализ данных

Перед обучением модели изучаем данные:

1. баланс классов и пропуски;
2. распределения числовых признаков;
3. частоты категорий и долю согласий в каждой группе;
4. различия между согласившимися и отказавшимися;
5. связи между признаками.

Данные упорядочены по времени — это важно учесть при разбиении на выборки.


In [ ]:
cat_cols = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

y_counts = df['y'].value_counts()
sns.barplot(x=y_counts.index.astype(str), y=y_counts.values, ax=axes[0], palette='muted')
axes[0].set_title('Распределение целевой переменной')
axes[0].set_xlabel('y')
axes[0].set_ylabel('Количество')

conversion_by_month = df.groupby('month', observed=True)['y'].apply(lambda s: (s == 'yes').mean())
conversion_by_month = conversion_by_month.reindex(month_order)
conversion_by_month.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Доля согласий на вклад по месяцам')
axes[1].set_xlabel('month')
axes[1].set_ylabel('Доля согласий')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Пропуски:')
print(df.isna().sum())
print('\nДубликаты:', df.duplicated().sum())
print('\nЧисловые признаки:', num_cols)
print('Категориальные признаки:', cat_cols)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)

axes.ravel()[-1].axis('off')
plt.suptitle('Как распределены числовые признаки', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    order = month_order if col == 'month' else df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, color='steelblue')
    ax.set_title(f'Распределение: {col}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('')

for ax in axes[len(cat_cols):]:
    ax.axis('off')

plt.suptitle('Распределения всех категориальных признаков', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    rates = df.groupby(col, observed=True)['y'].apply(lambda s: (s == 'yes').mean())
    if col == 'month':
        rates = rates.reindex(month_order)
    else:
        rates = rates.sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Доля согласий по {col}')
    ax.set_ylabel('Доля согласий')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

for ax in axes[len(cat_cols):]:
    ax.axis('off')

plt.suptitle('Связь категориальных признаков с целевой переменной', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
n_cols_grid = 4
n_rows = int(np.ceil(len(num_cols) / n_cols_grid))
fig, axes = plt.subplots(n_rows, n_cols_grid, figsize=(16, n_rows * 3))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    for label, group in df.groupby('y', observed=True):
        legend_label = 'согласие' if str(label) == 'yes' else 'отказ'
        ax.hist(group[col], bins=25, alpha=0.55, label=legend_label, density=True)
    ax.set_title(col)
    ax.legend()

for ax in axes[len(num_cols):]:
    ax.axis('off')

plt.suptitle('Распределения числовых признаков по классам y', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.3)
plt.title('Матрица корреляций Пирсона (числовые признаки)')
plt.tight_layout()
plt.show()

corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
strong_corr = corr_pairs.reindex(corr_pairs['correlation'].abs().sort_values(ascending=False).index)
strong_corr.head(10)


In [ ]:
top_pairs = strong_corr.head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

sample = df.sample(n=min(4000, len(df)), random_state=RANDOM_STATE).copy()
sample['y_legend'] = sample['y'].astype(str).map({'yes': 'согласие', 'no': 'отказ'})

for ax, (_, row) in zip(axes, top_pairs.iterrows()):
    f1, f2 = row['feature_1'], row['feature_2']
    sns.scatterplot(data=sample, x=f1, y=f2, hue='y_legend', alpha=0.45, s=18, ax=ax)
    ax.set_title(f'{f1} vs {f2}\n(r = {row["correlation"]:.3f})')

plt.suptitle('Диаграммы рассеяния для наиболее связанных числовых признаков', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
pair_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'y']
sample_for_pair = df[pair_cols].sample(n=2000, random_state=RANDOM_STATE).copy()
sample_for_pair['y'] = sample_for_pair['y'].astype(str).map({'yes': 'согласие', 'no': 'отказ'})

g = sns.pairplot(
    sample_for_pair,
    hue='y',
    diag_kind='kde',
    corner=True,
    plot_kws={'alpha': 0.35, 's': 15},
)
g.fig.suptitle('Связи между ключевыми числовыми признаками (выборка 2000)', y=1.02)
plt.show()


In [ ]:
# Выбросы растягивают ось (особенно balance, duration, campaign, pdays, previous),
# поэтому для сравнения классов рисуем boxplot без fliers.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    sns.boxplot(
        data=df, x='y', y=col, hue='y', ax=ax,
        palette='Set2', legend=False, showfliers=False,
    )
    ax.set_title(f'{col} по классам y')
    ax.set_xlabel('y')
    ax.set_ylabel(col)

axes[-1].axis('off')
plt.suptitle('Числовые признаки по целевой переменной y (без выбросов)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


### Что видно после анализа

- **Классы несбалансированы:** отказов гораздо больше, чем согласий. Поэтому accuracy не используем.
- **Сильнее всего связаны с согласием:** месяц звонка, исход прошлой кампании, канал связи, род занятий, наличие ипотеки.
- **Длительность звонка** заметно отличается у согласившихся и отказавшихся, но в модель не включаем — эта информация недоступна до звонка.
- У баланса, числа звонков и прошлых контактов есть **редкие большие значения** — проверим два способа масштабирования числовых признаков.
- Значение **`pdays = -1`** означает «раньше не звонили», а не «прошло 0 дней» — для модели это отдельный смысл.


## 3. Подготовка данных

**Пропусков нет** — заполнять нечего.

**Что сделали с данными:**

1. **Исключили `duration`** — чтобы модель не использовала информацию, которой нет до звонка.

2. **Обработали `pdays = -1`:**
   - создали признак `was_contacted_before` (1 — звонили раньше, 0 — нет);
   - в числовом поле `pdays` заменили `-1` на `0`.

3. **Категориальные признаки** (работа, месяц, канал связи и др.) закодировали через one-hot — каждое значение стало отдельным столбцом.

4. **Числовые признаки** масштабировали — чтобы возраст, баланс и число звонков были в сопоставимом масштабе.

5. **Целевую переменную** перевели в числа: `yes` → 1, `no` → 0.

Все преобразования обучаются **только на обучающей выборке**, чтобы проверочная и тестовая не «подсказывали» ответ модели.


In [ ]:
DROP_COLS = ['duration']
CAT_COLS = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
BASE_NUM_COLS = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


def prepare_features(df_in: pd.DataFrame, use_was_contacted: bool = True) -> pd.DataFrame:
    """Подготовка данных без использования проверочной и тестовой выборок."""
    data = df_in.drop(columns=DROP_COLS, errors='ignore').copy()
    if use_was_contacted:
        data['was_contacted_before'] = (data['pdays'] != -1).astype(int)
    data.loc[data['pdays'] == -1, 'pdays'] = 0
    data['y_bin'] = (data['y'] == 'yes').astype(int)
    return data


df_prepared = prepare_features(df)
print(f'После подготовки: {df_prepared.shape[0]} строк, {df_prepared.shape[1]} колонок')
print(f'Доля yes: {df_prepared["y_bin"].mean():.1%}')
print(f'was_contacted_before == 1: {(df_prepared["was_contacted_before"] == 1).mean():.1%}')
df_prepared[CAT_COLS + BASE_NUM_COLS + ['was_contacted_before', 'y_bin']].head()


### Проверка: что из подготовки данных реально помогает

Сравниваем четыре варианта на **проверочной выборке** (20% данных):

| Способ масштабирования | Признак «был контакт раньше» |
|---|---|
| StandardScaler | да / нет |
| RobustScaler | да / нет |

**Тестовую выборку на этом этапе не используем** — она нужна только для финальной оценки.


In [ ]:
def temporal_split(data: pd.DataFrame, train_frac=0.6, val_frac=0.2):
    n = len(data)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return data.iloc[:train_end], data.iloc[train_end:val_end], data.iloc[val_end:]


def feature_lists(use_was_contacted: bool):
    cat = CAT_COLS.copy()
    num = BASE_NUM_COLS.copy()
    if use_was_contacted:
        num = num + ['was_contacted_before']
    return cat, num


def quick_logreg_score(scaler, use_was_contacted=True):
    data = prepare_features(df, use_was_contacted=use_was_contacted)
    train_df, val_df, _ = temporal_split(data)

    cat_cols, num_cols = feature_lists(use_was_contacted)
    feature_cols = cat_cols + num_cols

    X_train = train_df[feature_cols]
    y_train = train_df['y_bin']
    X_val = val_df[feature_cols]
    y_val = val_df['y_bin']

    pre = ColumnTransformer([
        ('num', scaler, num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ])
    pipe = Pipeline([
        ('pre', pre),
        ('model', LogisticRegression(
            max_iter=10_000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, proba)


ablation = pd.DataFrame([
    {
        'scaler': 'StandardScaler',
        'was_contacted_before': 'нет',
        'PR-AUC (val)': quick_logreg_score(StandardScaler(), False),
    },
    {
        'scaler': 'RobustScaler',
        'was_contacted_before': 'нет',
        'PR-AUC (val)': quick_logreg_score(RobustScaler(), False),
    },
    {
        'scaler': 'StandardScaler',
        'was_contacted_before': 'да',
        'PR-AUC (val)': quick_logreg_score(StandardScaler(), True),
    },
    {
        'scaler': 'RobustScaler',
        'was_contacted_before': 'да',
        'PR-AUC (val)': quick_logreg_score(RobustScaler(), True),
    },
])
ablation = ablation.sort_values('PR-AUC (val)', ascending=False).reset_index(drop=True)
ablation


По таблице выбираем лучшую комбинацию масштабирования и признаков. Столбец «был контакт раньше» показывает, помогает ли новый признак — остальные шаги одинаковы.


## 4. Разделение данных

Записи упорядочены по дате, поэтому делим **по времени**, а не случайно:

| Выборка | Доля | Зачем |
|---|---|---|
| **Обучающая (train)** | 60% | обучение модели |
| **Проверочная (validation)** | 20% | выбор настроек и сравнение вариантов |
| **Тестовая (test)** | 20% | финальная оценка один раз |

Так модель учится на «прошлом» и проверяется на более поздних кампаниях — ближе к реальной задаче.


In [ ]:
# Выбираем лучший вариант подготовки данных по проверочной выборке
best_row = ablation.iloc[0]
USE_WAS_CONTACTED = best_row['was_contacted_before'] == 'да'
CHOSEN_SCALER_NAME = best_row['scaler']

print(f'Выбран вариант: {CHOSEN_SCALER_NAME}, признак «был контакт раньше» = {USE_WAS_CONTACTED}')
print(f'PR-AUC на проверочной выборке: {best_row["PR-AUC (val)"]:.4f}')

model_df = prepare_features(df, use_was_contacted=USE_WAS_CONTACTED)
CAT_FEATURES, NUM_FEATURES = feature_lists(USE_WAS_CONTACTED)
FEATURE_COLUMNS = CAT_FEATURES + NUM_FEATURES

train_df, val_df, test_df = temporal_split(model_df)

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['y_bin']
X_val = val_df[FEATURE_COLUMNS]
y_val = val_df['y_bin']
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df['y_bin']

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(
        f'{name:5} | rows={len(part):5} | '
        f"yes={part['y_bin'].mean():.1%} | month={part['month'].iloc[0]}..{part['month'].iloc[-1]}"
    )
print(f'Числовых признаков: {len(NUM_FEATURES)}, категориальных: {len(CAT_FEATURES)}')


## 5. Обучение моделей

Сравниваем **четыре варианта логистической регрессии**:

| Модель | Смысл |
|---|---|
| Без регуляризации | базовый вариант |
| L2 (Ridge) | сглаживает веса, все признаки остаются |
| L1 (Lasso) | может «отключить» слабые признаки |
| ElasticNet | комбинация L1 и L2 |

У всех моделей включён **баланс классов** — чтобы редкие согласия не игнорировались.

Обработка данных и модель объединены в один **конвейер (Pipeline)**: сначала преобразование признаков, затем предсказание. Так исключается утечка данных между этапами.


In [ ]:
def build_preprocessor():
    scaler = RobustScaler() if CHOSEN_SCALER_NAME == 'RobustScaler' else StandardScaler()
    return ColumnTransformer([
        ('num', scaler, NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ])


def collect_metrics(model, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y, proba),
        'pr_auc': average_precision_score(y, proba),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
    }


def metrics_table(model, splits, threshold=0.5):
    rows = []
    for split_name, (X_split, y_split) in splits.items():
        metrics = collect_metrics(model, X_split, y_split, threshold=threshold)
        metrics['split'] = split_name
        rows.append(metrics)
    return pd.DataFrame(rows).set_index('split')[['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']]


def evaluate_model(name, model, param_grid=None):
    pipe = Pipeline([
        ('pre', build_preprocessor()),
        ('model', model),
    ])
    if param_grid is None:
        pipe.fit(X_train, y_train)
        best_pipe = pipe
        best_params = {}
    else:
        grid = GridSearchCV(
            pipe,
            param_grid=param_grid,
            scoring='average_precision',
            cv=3,
            n_jobs=-1,
        )
        grid.fit(X_train, y_train)
        best_pipe = grid.best_estimator_
        best_params = grid.best_params_

    val_metrics = collect_metrics(best_pipe, X_val, y_val)
    test_metrics = collect_metrics(best_pipe, X_test, y_test)
    return {
        'model': name,
        'best_params': best_params,
        'PR-AUC (val)': val_metrics['pr_auc'],
        'ROC-AUC (val)': val_metrics['roc_auc'],
        'F1 (val)': val_metrics['f1'],
        'PR-AUC (test)': test_metrics['pr_auc'],
        'ROC-AUC (test)': test_metrics['roc_auc'],
        'F1 (test)': test_metrics['f1'],
        'pipeline': best_pipe,
    }


splits = {
    'train': (X_train, y_train),
    'val': (X_val, y_val),
    'test': (X_test, y_test),
}


In [ ]:
C_GRID = np.logspace(-3, 2, 8)

results = [
    evaluate_model(
        'Без регуляризации',
        LogisticRegression(
            penalty=None,
            solver='lbfgs',
            max_iter=10_000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
    ),
    evaluate_model(
        'L2-регуляризация',
        LogisticRegression(
            penalty='l2',
            solver='lbfgs',
            max_iter=10_000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
        param_grid={'model__C': C_GRID},
    ),
    evaluate_model(
        'L1-регуляризация',
        LogisticRegression(
            penalty='l1',
            solver='saga',
            max_iter=10_000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
        param_grid={'model__C': C_GRID},
    ),
    evaluate_model(
        'ElasticNet',
        LogisticRegression(
            penalty='elasticnet',
            solver='saga',
            max_iter=10_000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
        param_grid={
            'model__C': C_GRID,
            'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
        },
    ),
]

metrics_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in ('pipeline', 'best_params')} for r in results]
)
metrics_df = metrics_df.sort_values('PR-AUC (val)', ascending=False).reset_index(drop=True)
metrics_df


In [ ]:
for r in results:
    print(f"{r['model']}: {r['best_params'] or '(параметры по умолчанию)'}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = sns.color_palette('muted', n_colors=4)

for ax, metric, title in zip(
    axes,
    ['PR-AUC (test)', 'ROC-AUC (test)', 'F1 (test)'],
    ['PR-AUC', 'ROC-AUC', 'F1'],
):
    plot_df = metrics_df.sort_values(metric, ascending=False)
    sns.barplot(data=plot_df, x='model', y=metric, ax=ax, palette=colors)
    ax.set_title(title)
    ax.set_xlabel('Модель')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Сравнение моделей на тестовой выборке', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
def get_feature_names(preprocessor):
    cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
    return list(NUM_FEATURES) + list(cat_names)


def plot_feature_importance(pipeline, model_name, top_n=20):
    pre = pipeline.named_steps['pre']
    coefs = pipeline.named_steps['model'].coef_.ravel()
    feature_names = get_feature_names(pre)

    importance = pd.Series(np.abs(coefs), index=feature_names).sort_values(ascending=False)
    top = importance.head(top_n).sort_values()

    plt.figure(figsize=(10, 8))
    top.plot(kind='barh', color='steelblue')
    plt.title(f'Наиболее влиятельные признаки ({model_name})')
    plt.xlabel('Сила влияния признака')
    plt.tight_layout()
    plt.show()
    return importance


best_result = max(results, key=lambda r: r['PR-AUC (val)'])
final_model = best_result['pipeline']
final_name = best_result['model']
print(f'Финальная модель (лучшая на проверочной выборке): {final_name}')
print(f"PR-AUC val={best_result['PR-AUC (val)']:.3f}, test={best_result['PR-AUC (test)']:.3f}")

importance = plot_feature_importance(final_model, final_name)
importance.head(15)


In [ ]:
y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[0].plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_test_proba):.3f}')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_xlabel('Доля ложноположительных (FPR)')
axes[0].set_ylabel('Доля истинноположительных (TPR)')
axes[0].set_title(f'ROC-кривая на тестовой выборке ({final_name})')
axes[0].legend()

PrecisionRecallDisplay.from_predictions(y_test, y_test_proba, ax=axes[1], name=final_name)
axes[1].set_title(f'PR-кривая на тестовой выборке (AP = {average_precision_score(y_test, y_test_proba):.3f})')

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_test_pred, target_names=['no', 'yes']))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    xticklabels=['Предсказано: отказ', 'Предсказано: согласие'],
    yticklabels=['Факт: отказ', 'Факт: согласие'],
)
plt.xlabel('Предсказание')
plt.ylabel('Факт')
plt.title(f'Матрица ошибок на тестовой выборке ({final_name})')
plt.tight_layout()
plt.show()

print('Метрики финальной модели на всех выборках:')
metrics_table(final_model, splits)


## 6. Как избежали переобучения

**Переобучение** — когда модель хорошо работает на знакомых данных, но плохо на новых.

Что сделали:

1. **Разделение по времени** — тест содержит более поздние кампании.
2. **Конвейер обработки** — все преобразования обучаются только на train.
3. **Проверочная выборка** — выбор scaler и модели без участия test.
4. **Регуляризация** — ограничение «слишком больших» весов у модели.
5. **Подбор параметров** — только на train, выбор лучшего варианта — по validation.

Признак переобучения — большой разрыв метрик между train и test.


In [ ]:
overfit_check = metrics_table(final_model, splits)[['roc_auc', 'pr_auc', 'f1']]
train_test_gap = (
    overfit_check.loc['train', 'roc_auc'] - overfit_check.loc['test', 'roc_auc']
)
train_val_gap = (
    overfit_check.loc['train', 'roc_auc'] - overfit_check.loc['val', 'roc_auc']
)
print(f'Разрыв ROC-AUC train − val:  {train_val_gap:.3f}')
print(f'Разрыв ROC-AUC train − test: {train_test_gap:.3f}')
overfit_check


## 7. Выводы

1. **Задача:** по данным о клиенте и кампании предсказать согласие на срочный вклад.

2. **Данные:** 45 211 звонков, согласий ~12%. Accuracy не подходит — модель «всегда нет» выглядела бы неплохо, но была бы бесполезна.

3. **Подготовка:** исключили длительность звонка (утечка данных); создали признак «был ли контакт раньше»; сравнили два способа масштабирования.

4. **Разделение:** 60% / 20% / 20% по времени — train, validation, test.

5. **Модели:** четыре варианта логистической регрессии. Лучший на validation — **L2 с регуляризацией**.

6. **Результаты на test (финальная модель L2):**
   - PR-AUC ≈ 0.39 (заметно выше базового уровня ~0.12);
   - recall ≈ 55% — находим больше половины согласившихся;
   - precision ≈ 38% — из предсказанных «да» около 38% действительно соглашаются;
   - F1 ≈ 0.45.

7. **Важные признаки:** месяц звонка (may, oct), семейное положение, канал связи, возраст, число звонков в кампании.

8. **Переобучение:** разрыв ROC-AUC между train и test небольшой (~0.02) — модель ведёт себя стабильно.

**Практический смысл:** модель помогает **ранжировать клиентов** — кому звонить в первую очередь. Полностью предсказать согласие нельзя (данных до звонка мало), но приоритизация обзвона уже полезна.
